In [5]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import StandardCNN, DropoutCNN, BatchNormCNN, FullRegularizedCNN
from src.data import get_torch_dataloaders, ClassDistributionAnalyzer
from src.training import RegularizationExperimentSuite

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute Device: {device}')

Compute Device: cpu


In [6]:
# Analyze class distribution
analyzer = ClassDistributionAnalyzer()
stats = analyzer.analyze()
print(f"Total Train Images: {stats['total_train']} | Test Images: {stats['total_test']}")
print(f"Imbalance Ratio: {stats['imbalance_ratio']}x")

# Non-Augmented DataLoaders (Baseline)
standard_loaders = get_torch_dataloaders(batch_size=64, augment_train=False)
# Augmented DataLoaders
augmented_loaders = get_torch_dataloaders(batch_size=64, augment_train=True)

print("DataLoaders successfully prepared!")

Total Train Images: 28716 | Test Images: 7185
Imbalance Ratio: 16.51x
DataLoaders successfully prepared!


In [7]:
m_standard = StandardCNN()
m_dropout = DropoutCNN()
m_bn = BatchNormCNN()
m_full = FullRegularizedCNN()

print(f"Standard CNN (No Reg) Params : {m_standard.get_num_params()[0]:,}")
print(f"Dropout CNN Params           : {m_dropout.get_num_params()[0]:,}")
print(f"BatchNorm CNN Params         : {m_bn.get_num_params()[0]:,}")
print(f"Full Regularized CNN Params  : {m_full.get_num_params()[0]:,}")

Standard CNN (No Reg) Params : 1,500,135
Dropout CNN Params           : 1,500,135
BatchNorm CNN Params         : 1,501,799
Full Regularized CNN Params  : 1,501,799


In [ ]:
# Initialize Experiment Suite
suite = RegularizationExperimentSuite(batch_size=64, lr=0.001)

# Run all experiments (Set epochs=10 for full training run)
results = suite.run_all(epochs=10, patience=4)

Preparing DataLoaders for Regularization Experiments...

          STARTING REGULARIZATION EXPERIMENTS SUITE
  Target Experiments: Baseline_NoReg, Dropout_Only, BatchNorm_Only, Augmentation_Only, Full_Regularization
  Epochs per Experiment: 10 | Patience: 4

##############################################################################
  RUNNING EXPERIMENT: 1. Baseline (No Regularization)
  Description: No Dropout, No BatchNorm, No Augmentation
##############################################################################
       MODEL CONFIGURATION: STANDARD_NOREG_CNN
 Batch Normalization : Disabled
 Dropout             : Disabled (Conv: 0.0, Dense: 0.0)
 Trainable Params    : 1,500,135
 Total Params        : 1,500,135

  STARTING MODEL TRAINING: EXP_BASELINE_NOREG (10 Epochs)
  Device: cpu | Train Batches: 359 | Val Batches: 90
Epoch [01/10] Train Loss: 1.9463 | Train Acc: 16.32% | Val Loss: 1.9451 | Val Acc: 17.29% | LR: 0.001000 (319.9s)  --> [BEST MODEL SAVED]
  --> EarlyStopping c

In [2]:
# Display Comparative Summary Table
summary_df = suite.get_summary_dataframe()
summary_df

NameError: name 'suite' is not defined